# 🧪 W10-D7 Virtual CTO Review：从观察到优先级

> 配套阅读：同名 `.md`。本 notebook 只用小规模、可重复的模拟来验证核心治理约束。

**实验目标：** 用可复算指标汇总本周治理观察，并检验“先修 PII 默认值”的决策是否稳健。


In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

# 本周主题评分（教学用、可调整）：评分不是事实结论，而是把判断显式化。
topics = ["Permission", "Audit/Trace", "Prompt custody", "Fail-closed", "Rollback/FEC", "Coverage map"]
scores = np.array([8.5, 8.0, 9.0, 8.5, 8.5, 9.0])
print("本周理解均值:", round(scores.mean(), 2), "/ 10")
print("最低项（下一轮复习优先）:", topics[scores.argmin()])


In [ ]:
weeks = ["W8", "W9", "W10"]
quality = np.array([7.2, 6.8, 6.8])
understanding = np.array([7.0, 7.5, 8.5])
plt.figure(figsize=(7, 3.5))
plt.plot(weeks, quality, marker="o", label="工程综合评分", color="#4C78A8")
plt.plot(weeks, understanding, marker="o", label="理解深度", color="#59A14F")
plt.ylim(5.5, 10); plt.ylabel("分数 / 10"); plt.title("复盘：理解提升不等于工程缺口已消失")
plt.legend(); plt.grid(axis="y", alpha=.25); plt.tight_layout(); plt.show()


In [ ]:
# CTO 决策模型：用不同权重做敏感性分析，避免只凭直觉。
issues = {
    "PII 默认关闭": (10, 10, 1),
    "出站签名缺失": (8, 7, 6),
    "跨租户测试不足": (8, 6, 5),
    "合规报告缺失": (5, 3, 5),
}
weight_sets = [(1, 1, 1), (1.2, 1, 1), (1, 1.2, 1), (1, 1, .7)]
for wi, wu, wc in weight_sets:
    ranks = {name: (impact*wi + urgency*wu) / (cost*wc) for name, (impact, urgency, cost) in issues.items()}
    winner = max(ranks, key=ranks.get)
    print(f"权重 impact={wi}, urgency={wu}, cost={wc} -> 第一优先：{winner}")

print("结论：在四组合理权重下，PII 默认关闭均为第一优先，适合作为 P0 修复项。")
